In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from keras.models import Sequential
from keras.layers import SimpleRNN, Dense, Dropout
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping
from sklearn.model_selection import TimeSeriesSplit

def smape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return 100 * np.mean(
        2 * np.abs(y_pred - y_true) /
        (np.abs(y_true) + np.abs(y_pred))
    )

def mase(y_true, y_pred, y_train):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_train = np.array(y_train)

    naive_error = np.mean(np.abs(np.diff(y_train)))
    return np.mean(np.abs(y_true - y_pred)) / naive_error

def theil_u(y_true, y_pred):
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    denominator = np.sqrt(np.mean(y_true ** 2) + np.mean(y_pred ** 2))
    u = rmse / denominator
    return u

# Load data
df = pd.read_excel('output_week.xlsx')

# Count occurrences and sort by week
data = df['WEEK'].value_counts().reset_index()
data.columns = ['WEEK', 'DATA']

# Convert week to a date representing the beginning of the week
data['WEEK_period'] = pd.to_datetime(
    data['WEEK'] + '-1',
    format='%G-%V-%u',
    errors='coerce'
)
data = data.sort_values('WEEK_period').reset_index(drop=True)

# ------------------------------
# Smoothing (moving average)
# ------------------------------
data['DATA_smooth'] = data['DATA'].rolling(window=5, center=True).mean()
data['DATA_smooth'].fillna(method='bfill', inplace=True)
data['DATA_smooth'].fillna(method='ffill', inplace=True)

# Data normalization
scaler = MinMaxScaler(feature_range=(0.1, 1))
data['y'] = scaler.fit_transform(data[['DATA_smooth']])

# ------------------------------
# Create lag variables
# ------------------------------
n_lags = 11  # You may test 12, 16, or 26 lags

for i in range(1, n_lags + 1):
    data[f'lag_{i}'] = data['y'].shift(i)

data = data.dropna().reset_index(drop=True)

# ------------------------------
# Prepare X and y
# ------------------------------
lag_cols = [f'lag_{i}' for i in range(n_lags, 0, -1)]

X = data[lag_cols].values.astype(np.float32)
y = data['y'].values.astype(np.float32)

# Reshape input for RNN
X = X.reshape((X.shape[0], X.shape[1], 1))

/var/folders/36/lb3mdhtj2lldbybnz877q_hr0000gn/T/ipykernel_2583/687632434.py:53: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['DATA_smooth'].fillna(method='bfill', inplace=True)
/var/folders/36/lb3mdhtj2lldbybnz877q_hr0000gn/T/ipykernel_2583/687632434.py:53: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data['DATA_smooth'].fillna(method='bfill', inplace=True)
/var/folders/36/lb3mdhtj2lldbybnz877q_hr0000gn/T/ipykernel_2583/687632434.p

In [3]:
tscv = TimeSeriesSplit(n_splits=10)


rmse_rnn = []
mae_rnn = []
mase_rnn = []
smape_rnn = []


for fold, (train_idx, test_idx) in enumerate(tscv.split(X), 1):

    print(f"\nFold {fold}")

    # ------------------------------
    # Split temporal
    # ------------------------------

    X_train = X[train_idx]
    y_train = y[train_idx]

    X_test = X[test_idx]
    y_test = y[test_idx]

    # ------------------------------
    # Simple RNN Model
    # ------------------------------

    model = Sequential()

    model.add(
        SimpleRNN(
            units=128,
            activation='tanh',
            return_sequences=False,
            input_shape=(n_lags,1)
        )
    )

    model.add(Dense(1))


    # ------------------------------
    # Compile
    # ------------------------------

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='mse'
    )


    # ------------------------------
    # Early stopping
    # ------------------------------

    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True
    )


    # ------------------------------
    # Training
    # ------------------------------

    history = model.fit(
        X_train,
        y_train,
        validation_split=0.2,
        epochs=200,
        batch_size=16,
        callbacks=[early_stop],
        verbose=0
    )


    # ------------------------------
    # Forecast
    # ------------------------------

    y_pred = model.predict(
        X_test,
        verbose=0
    ).flatten()


    # ------------------------------
    # Return to original scale
    # ------------------------------

    y_test_real = scaler.inverse_transform(
        y_test.reshape(-1,1)
    ).flatten()


    y_pred_real = scaler.inverse_transform(
        y_pred.reshape(-1,1)
    ).flatten()


    y_train_real = scaler.inverse_transform(
        y_train.reshape(-1,1)
    ).flatten()


    # ------------------------------
    # Metrics
    # ------------------------------

    mase_value = mase(
        y_test_real,
        y_pred_real,
        y_train_real
    )


    rmse_value = np.sqrt(
        mean_squared_error(
            y_test_real,
            y_pred_real
        )
    )


    mae_value = mean_absolute_error(
        y_test_real,
        y_pred_real
    )


    smape_value = smape(
        y_test_real,
        y_pred_real
    )


    # ------------------------------
    # Store results
    # ------------------------------

    rmse_rnn.append(rmse_value)
    mae_rnn.append(mae_value)
    mase_rnn.append(mase_value)
    smape_rnn.append(smape_value)


    print(
        f"RMSE={rmse_value:.4f} | "
        f"MAE={mae_value:.4f}"
    )



# ------------------------------
# Save results
# ------------------------------

results_rnn = pd.DataFrame({

    "Fold": range(1,len(rmse_rnn)+1),
    "RMSE": rmse_rnn,
    "MAE": mae_rnn,
    "MASE": mase_rnn,
    "SMAPE": smape_rnn

})


results_rnn.to_csv(
    "RNN_results.csv",
    index=False
)


Fold 1


/Users/octaciliofsilvajunior/miniforge3/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


RMSE=4.5241 | MAE=3.7143

Fold 2


/Users/octaciliofsilvajunior/miniforge3/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


RMSE=3.6968 | MAE=2.9901

Fold 3


...

RMSE=0.8066 | MAE=0.6417

Fold 9


/Users/octaciliofsilvajunior/miniforge3/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


RMSE=0.9248 | MAE=0.7875

Fold 10


/Users/octaciliofsilvajunior/miniforge3/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


RMSE=0.7366 | MAE=0.6081
